# Milestone 3 - Weak Scaling (MPI GMM)

Uses Milestone 2 distributed GMM as blueprint with Milestone 3 instrumentation.

Weak scaling: fixed rows per rank, increasing MPI ranks.
Measured metrics: runtime trend, weak-scale factor, efficiency, communication overhead, iteration behavior, convergence, memory, I/O, and quality/coherence metrics.

## AWS EC2 Tips

- Keep the same EC2 type across all weak-scaling runs.
- Keep BLAS/OpenMP thread vars fixed to 1.
- For larger rank counts, use enough vCPUs (or multi-node MPI with proper hostfile and networking).

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

print(PROJECT_ROOT)

In [ ]:
EMBEDDING = "bge"
DISTRIBUTION = "block"
N_CLUSTERS = 10
ROWS_PER_RANK = 1000
RANKS = [1, 2, 4, 8, 16]
N_INIT = 1
MAX_ITER = 120

OUT_CSV = RESULTS_DIR / "m3_weak_scaling_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(ranks):
    cmd = [
        "mpirun", "-np", str(ranks),
        "python", str(RUNNER),
        "--mode", "weak",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(N_CLUSTERS),
        "--rows-per-rank", str(ROWS_PER_RANK),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tag", f"weak_p{ranks}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for p in RANKS:
    run_mpi_experiment(p)

print("Weak scaling runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("mpi_ranks").reset_index(drop=True)
baseline_time = float(df.loc[df["mpi_ranks"].idxmin(), "gmm_total_seconds"])

df["weak_scale_factor"] = df["gmm_total_seconds"] / baseline_time
df["weak_efficiency"] = baseline_time / df["gmm_total_seconds"]
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "mpi_ranks", "rows_per_rank_target", "rows_used",
    "gmm_total_seconds", "weak_scale_factor", "weak_efficiency",
    "avg_iteration_seconds", "communication_seconds", "comm_overhead_pct",
    "gmm_iterations", "convergence_rate_iter_per_s",
    "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
print(df[display_cols].to_string(index=False))

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["mpi_ranks"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Weak Scaling Runtime")
axes[0, 0].set_xlabel("MPI ranks")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["mpi_ranks"], df["weak_scale_factor"], marker="o", label="Measured")
axes[0, 1].axhline(1.0, linestyle="--", label="Ideal")
axes[0, 1].set_title("Weak Scale Factor")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df["mpi_ranks"], 100.0 * df["weak_efficiency"], marker="o")
axes[1, 0].set_title("Weak Scaling Efficiency (%)")
axes[1, 0].set_xlabel("MPI ranks")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["mpi_ranks"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["mpi_ranks"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Communication and Iteration")
axes[1, 1].set_xlabel("MPI ranks")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_weak_scaling_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

# Milestone 3 - Weak Scaling (MPI GMM)

This notebook runs **weak scaling** using the Milestone 2 GMM blueprint with Milestone 3 instrumentation.

Weak scaling setup:
- Fixed rows per rank
- Increase MPI ranks
- Total rows scale proportionally with ranks

Measured metrics include runtime, per-iteration behavior, communication overhead, convergence, memory, I/O, and clustering quality/coherence.

## AWS EC2 Tips For Weak Scaling

- Prefer one EC2 type for all runs to avoid hardware variation.
- Keep CPU pinning and BLAS thread vars fixed (`*_NUM_THREADS=1`).
- If testing many ranks on one node, ensure the instance has at least that many vCPUs.
- For multi-node MPI, configure hostfile and security group rules for MPI communication.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

plt.rcParams["figure.dpi"] = 120

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

In [ ]:
# Experiment configuration
EMBEDDING = "bge"
DISTRIBUTION = "block"
N_CLUSTERS = 10
ROWS_PER_RANK = 1000
RANKS = [1, 2, 4, 8]
N_INIT = 1
MAX_ITER = 120

OUT_CSV = RESULTS_DIR / "m3_weak_scaling_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(ranks):
    cmd = [
        "mpirun", "-np", str(ranks),
        "python", str(RUNNER),
        "--mode", "weak",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(N_CLUSTERS),
        "--rows-per-rank", str(ROWS_PER_RANK),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tag", f"weak_p{ranks}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for p in RANKS:
    run_mpi_experiment(p)

print("Weak scaling runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("mpi_ranks").reset_index(drop=True)
baseline_time = float(df.loc[df["mpi_ranks"].idxmin(), "gmm_total_seconds"])

# Weak scaling ideal: runtime approximately constant
df["weak_scale_factor"] = df["gmm_total_seconds"] / baseline_time
df["weak_efficiency"] = baseline_time / df["gmm_total_seconds"]
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "mpi_ranks", "rows_per_rank_target", "rows_used",
    "gmm_total_seconds", "weak_scale_factor", "weak_efficiency",
    "avg_iteration_seconds", "communication_seconds", "comm_overhead_pct",
    "gmm_iterations", "convergence_rate_iter_per_s",
    "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
display(df[display_cols])

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["mpi_ranks"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Weak Scaling Runtime")
axes[0, 0].set_xlabel("MPI ranks")
axes[0, 0].set_ylabel("seconds")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["mpi_ranks"], df["weak_scale_factor"], marker="o", label="Measured")
axes[0, 1].axhline(1.0, linestyle="--", label="Ideal")
axes[0, 1].set_title("Weak Scale Factor (T_p / T_1)")
axes[0, 1].set_xlabel("MPI ranks")
axes[0, 1].set_ylabel("factor")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df["mpi_ranks"], 100 * df["weak_efficiency"], marker="o")
axes[1, 0].set_title("Weak Scaling Efficiency")
axes[1, 0].set_xlabel("MPI ranks")
axes[1, 0].set_ylabel("efficiency %")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["mpi_ranks"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["mpi_ranks"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Communication and Iteration Metrics")
axes[1, 1].set_xlabel("MPI ranks")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_weak_scaling_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

## Notes

- In weak scaling, the target is nearly flat runtime as ranks increase.
- If runtime rises quickly, inspect `comm_overhead_pct` and switch distribution strategy (`block`, `cyclic`, `load_balanced`).